<a href="https://colab.research.google.com/github/yhkovska/A-B-Test-Conversion-Lift-Statistical-Significance/blob/main/A_B_Testing_Statistical_Analysis_with_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **A/B Testing Statistical Analysis with Python**

### **Project Overview**

The goal of this project is to analyze A/B testing results using statistical hypothesis testing in Python and prepare a dataset for visualization in Tableau.

The project includes:

- extracting A/B testing data from BigQuery;
- calculating four conversion metrics;
- performing statistical significance testing using a two-proportion z-test;
- comparing Control and Test groups;
- exporting the final dataset for Tableau dashboard creation.

The analyzed metrics are:

- add_payment_info / session
- add_shipping_info / session
- begin_checkout / session
- new_accounts / session

### **Data Preparation**

In [ ]:
# import libraries
from google.colab import auth
from google.colab import files
from google.cloud import bigquery


import pandas as pd
import numpy as np

from statsmodels.stats.proportion import proportions_ztest
import statsmodels.api as sm

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

In [ ]:
# authentification
!pip install --upgrade google-cloud-bigquery -q
auth.authenticate_user()
project_id = "data-analytics-mate"
client = bigquery.Client(project=project_id)

print(f"Connected to project: {project_id}")

Connected to project: data-analytics-mate


In [ ]:
 # SQL query
query = """
WITH
  session_info AS (
    SELECT
      s.date,
      s.ga_session_id,
      sp.country,
      sp.device,
      sp.continent,
      sp.channel,
      ab.test,
      ab.test_group
    FROM `DA.ab_test` ab
    JOIN `DA.session` s
      ON ab.ga_session_id = s.ga_session_id
    JOIN `DA.session_params` sp
      ON ab.ga_session_id = sp.ga_session_id
  ),
  session_with_orders AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT o.ga_session_id) AS session_with_orders
    FROM `DA.order` o
    JOIN session_info
      ON o.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  ),
  events AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name,
      COUNT(ep.ga_session_id) AS event_cnt
    FROM `DA.event_params` ep
    JOIN session_info
      ON ep.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name
  ),
  sessions AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
    FROM session_info
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  ),
  account AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
    FROM `DA.account_session` acs
    JOIN session_info
      ON acs.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  )
SELECT
  session_with_orders.date,
  session_with_orders.country,
  session_with_orders.device,
  session_with_orders.continent,
  session_with_orders.channel,
  session_with_orders.test,
  session_with_orders.test_group,
  'session_with_orders' AS event_name,
  session_with_orders.session_with_orders AS value
FROM session_with_orders
UNION ALL
SELECT
  events.date,
  events.country,
  events.device,
  events.continent,
  events.channel,
  events.test,
  events.test_group,
  event_name,
  event_cnt AS value
FROM events
UNION ALL
SELECT
  sessions.date,
  sessions.country,
  sessions.device,
  sessions.continent,
  sessions.channel,
  sessions.test,
  sessions.test_group,
  'sessions' AS event_name,
  session_cnt AS value
FROM sessions
UNION ALL
SELECT
  account.date,
  account.country,
  account.device,
  account.continent,
  account.channel,
  account.test,
  account.test_group,
  'new account' AS event_name,
  new_account_cnt AS value
FROM account
"""

In [ ]:
# load dataframe
# df = client.query(query).to_dataframe()
# display(df.head())
# print(df.shape)

df = client.query(query).to_dataframe(create_bqstorage_client=False)

display(df.head())
print(df.shape)

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-04,Myanmar (Burma),mobile,Asia,Organic Search,2,2,new account,1
1,2020-11-04,Palestine,mobile,Asia,Organic Search,2,1,new account,1
2,2020-11-06,Slovenia,desktop,Europe,Organic Search,2,2,new account,1
3,2020-11-07,Palestine,desktop,Asia,Paid Search,2,2,new account,1
4,2020-11-08,Mongolia,desktop,Asia,Undefined,2,2,new account,1


(800996, 9)


In [ ]:
# clean event names
df['event_name'] = (df['event_name'].str.strip().str.lower())

In [ ]:
# data aggregation (events)
aggregated_data = (df.groupby(
    ['test',
     'test_group',
     'country',
     'device',
     'continent',
     'channel',
     'event_name'],
    as_index=False)
['value']
.sum())

display(aggregated_data.head())

,test,test_group,country,device,continent,channel,event_name,value
0,1,1,(not set),desktop,(not set),Direct,first_visit,11
1,1,1,(not set),desktop,(not set),Direct,new account,3
2,1,1,(not set),desktop,(not set),Direct,page_view,35
3,1,1,(not set),desktop,(not set),Direct,scroll,15
4,1,1,(not set),desktop,(not set),Direct,session_start,15


In [ ]:
# final check
print(f'Unique event names: {df.event_name.unique()}\n')

print(f'Unique tests: {df.test.unique()}\n')

print(f'Unique test groups: {df.test_group.unique()}\n')

Unique event names: ['new account' 'session_with_orders' 'sessions' 'page_view' 'view_item'
 'first_visit' 'user_engagement' 'add_shipping_info' 'session_start'
 'scroll' 'view_promotion' 'begin_checkout' 'select_item' 'add_to_cart'
 'view_search_results' 'select_promotion' 'add_payment_info' 'click'
 'view_item_list']

Unique tests: <IntegerArray>
[2, 1, 4, 3]
Length: 4, dtype: Int64

Unique test groups: <IntegerArray>
[2, 1]
Length: 2, dtype: Int64



### **Statistical Significance Analysis**

In [ ]:
# metrics configuration
metrics_config = {
    'add_payment_info / session':  {'numerator_event': 'add_payment_info',  'denominator_event': 'sessions'},
    'add_shipping_info / session': {'numerator_event': 'add_shipping_info', 'denominator_event': 'sessions'},
    'begin_checkout / session':    {'numerator_event': 'begin_checkout',    'denominator_event': 'sessions'},
    'new_accounts / session':      {'numerator_event': 'new account',       'denominator_event': 'sessions'},
}

# additional dimensions for significance analysis
breakdown_dims = ['country', 'device', 'continent', 'channel']

In [ ]:
# test / group / slice / event / value table
def build_slices(df: pd.DataFrame, dims: list) -> pd.DataFrame:

    slices = []

    # test total
    overall = df.groupby(['test', 'test_group', 'event_name'], as_index=False)['value'].sum()
    overall['slice_dim'] = 'overall'
    overall['slice_value'] = 'overall'
    slices.append(overall)

    # additional slices
    for dim in dims:
        part = (
            df.groupby(['test', 'test_group', dim, 'event_name'], as_index=False)['value']
              .sum()
              .rename(columns={dim: 'slice_value'})
        )
        part['slice_dim'] = dim
        slices.append(part)

    result = pd.concat(slices, ignore_index=True)
    return result[['test', 'test_group', 'slice_dim', 'slice_value', 'event_name', 'value']]


slices_df = build_slices(aggregated_data, breakdown_dims)
display(slices_df.head())
print(slices_df.shape)

,test,test_group,slice_dim,slice_value,event_name,value
0,1,1,overall,overall,add_payment_info,1988
1,1,1,overall,overall,add_shipping_info,3034
2,1,1,overall,overall,add_to_cart,1395
3,1,1,overall,overall,begin_checkout,3784
4,1,1,overall,overall,click,368


(15720, 6)


In [ ]:
# pivot table for z-test
pivot_df = (
    slices_df
    .pivot_table(
        index=['test', 'test_group', 'slice_dim', 'slice_value'],
        columns='event_name',
        values='value',
        aggfunc='sum',
        fill_value=0,
    )
    .reset_index()
)
display(pivot_df.head())

event_name,test,test_group,slice_dim,slice_value,add_payment_info,add_shipping_info,add_to_cart,begin_checkout,click,first_visit,new account,page_view,scroll,select_item,select_promotion,session_start,session_with_orders,sessions,user_engagement,view_item,view_item_list,view_promotion,view_search_results
0,1,1,channel,Direct,392,664,269,823,79,7432,913,44951,17041,104,316,10875,1084,10691,40283,14644,6,6985,868
1,1,1,channel,Organic Search,640,1021,494,1249,90,12438,1307,69237,25561,207,442,15963,1502,15675,61525,22584,7,10274,1349
2,1,1,channel,Paid Search,450,770,379,967,91,9195,990,49040,18127,123,338,11981,1185,11777,43429,15638,8,7777,890
3,1,1,channel,Social Search,237,300,120,392,51,1492,312,15580,6638,61,101,3861,375,3883,14306,5168,2,2329,316
4,1,1,channel,Undefined,269,279,133,353,57,39,301,12735,5877,48,78,3225,368,3336,12245,4301,4,1823,255


In [ ]:
# cycle: z-test for each test / slice / metric
def get_control_test_labels(group_values):

    groups = list(pd.Series(group_values).unique())
    if len(groups) != 2:
        return None, None

    # string labels: identify the 'control' group.
    str_groups = [str(g).strip().lower() for g in groups]

    if 'control' in str_groups:
        idx = str_groups.index('control')
        control = groups[idx]
        test = groups[1 - idx]
    else:
        # numeric values: determine groups using sorted order.
        try:
            sorted_groups = sorted(groups)
        except TypeError:
            sorted_groups = sorted(groups, key=str)
        control, test = sorted_groups[0], sorted_groups[1]

    return control, test


results = []

for test_number in pivot_df['test'].unique():
    test_slice = pivot_df[pivot_df['test'] == test_number]

    for (slice_dim, slice_value), sub in test_slice.groupby(['slice_dim', 'slice_value']):
        control_label, test_label = get_control_test_labels(sub['test_group'].unique())
        if control_label is None:
            continue

        control_row = sub[sub['test_group'] == control_label]
        test_row = sub[sub['test_group'] == test_label]
        if control_row.empty or test_row.empty:
            continue

        for metric_name, cfg in metrics_config.items():
            num_event, den_event = cfg['numerator_event'], cfg['denominator_event']
            if num_event not in sub.columns or den_event not in sub.columns:
                continue

            numerator_control = control_row[num_event].values[0]
            denominator_control = control_row[den_event].values[0]
            numerator_test = test_row[num_event].values[0]
            denominator_test = test_row[den_event].values[0]

            if denominator_control == 0 or denominator_test == 0:
                continue

            conversion_control = numerator_control / denominator_control
            conversion_test = numerator_test / denominator_test

            # calculate the percentage change relative to the Control group.
            metric_change_pct = (
                (conversion_test - conversion_control) / conversion_control * 100
                if conversion_control != 0 else np.nan
            )

            # order [Treatment, Control] to keep the sign of z_stat consistent with metric_change_percent.
            count = np.array([numerator_test, numerator_control])
            nobs = np.array([denominator_test, denominator_control])
            try:
                z_stat, p_value = proportions_ztest(count, nobs)
            except Exception:
                z_stat, p_value = np.nan, np.nan

            results.append({
                'test_number': test_number,
                'slice_dim': slice_dim,
                'slice_value': slice_value,
                'metric': metric_name,
                'numerator_event': num_event,
                'denominator_event': den_event,
                'numerator_control': numerator_control,
                'denominator_control': denominator_control,
                'conversion_rate_control': conversion_control,
                'numerator_test': numerator_test,
                'denominator_test': denominator_test,
                'conversion_rate_test': conversion_test,
                'metric_change_pct': metric_change_pct,
                'z_stat': z_stat,
                'p_value': p_value,
                'significant': bool(p_value < 0.05) if pd.notna(p_value) else False,
            })

results_df = pd.DataFrame(results)

column_order = [
    'test_number', 'slice_dim', 'slice_value', 'metric',
    'numerator_event', 'denominator_event',
    'numerator_control', 'denominator_control', 'conversion_rate_control',
    'numerator_test', 'denominator_test', 'conversion_rate_test',
    'metric_change_pct', 'z_stat', 'p_value', 'significant',
]
results_df = results_df[column_order]

display(results_df.head())
print(results_df.shape)

/usr/local/lib/python3.13/dist-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std
/usr/local/lib/python3.13/dist-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std
/usr/local/lib/python3.13/dist-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std
/usr/local/lib/python3.13/dist-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std
/usr/local/lib/python3.13/dist-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std
/usr/local/lib/python3.13/dist-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std
/usr/local/lib/python3.13/dist-packages/statsm

,test_number,slice_dim,slice_value,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_pct,z_stat,p_value,significant
0,1,channel,Direct,add_payment_info / session,add_payment_info,sessions,392,10691,0.036666,516,10361,0.049802,35.825180,4.690261,0.000003,True
1,1,channel,Direct,add_shipping_info / session,add_shipping_info,sessions,664,10691,0.062108,716,10361,0.069105,11.265775,2.050707,0.040295,True
2,1,channel,Direct,begin_checkout / session,begin_checkout,sessions,823,10691,0.076981,915,10361,0.088312,14.719677,2.986589,0.002821,True
3,1,channel,Direct,new_accounts / session,new account,sessions,913,10691,0.085399,850,10361,0.082038,-3.935085,-0.879999,0.378860,False
4,1,channel,Organic Search,add_payment_info / session,add_payment_info,sessions,640,15675,0.040829,514,15631,0.032883,-19.461427,-3.730758,0.000191,True


(1968, 16)


In [ ]:
display(results_df[results_df['slice_dim'] == 'overall'])

output_path = 'ab_test_results.csv'
results_df.to_csv('ab_test_results.csv', index=False, sep=',', decimal='.')

,test_number,slice_dim,slice_value,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_pct,z_stat,p_value,significant
488,1,overall,overall,add_payment_info / session,add_payment_info,sessions,1988,45362,0.043825,2229,45193,0.049322,12.542021,3.924884,0.000087,True
489,1,overall,overall,add_shipping_info / session,add_shipping_info,sessions,3034,45362,0.066884,3221,45193,0.071272,6.560481,2.603571,0.009226,True
490,1,overall,overall,begin_checkout / session,begin_checkout,sessions,3784,45362,0.083418,4021,45193,0.088974,6.660587,2.978783,0.002894,True
491,1,overall,overall,new_accounts / session,new account,sessions,3823,45362,0.084278,3681,45193,0.081451,-3.354299,-1.542883,0.122859,False
980,2,overall,overall,add_payment_info / session,add_payment_info,sessions,2344,50637,0.046290,2409,50244,0.047946,3.576911,1.240994,0.214608,False
981,2,overall,overall,add_shipping_info / session,add_shipping_info,sessions,3480,50637,0.068724,3510,50244,0.069859,1.650995,0.709557,0.477979,False
982,2,overall,overall,begin_checkout / session,begin_checkout,sessions,4262,50637,0.084168,4313,50244,0.085841,1.988164,0.952898,0.340642,False
983,2,overall,overall,new_accounts / session,new account,sessions,4165,50637,0.082252,4184,50244,0.083274,1.241934,0.588793,0.556000,False
1472,3,overall,overall,add_payment_info / session,add_payment_info,sessions,3623,70047,0.051722,3697,70439,0.052485,1.474630,0.643172,0.520112,False
1473,3,overall,overall,add_shipping_info / session,add_shipping_info,sessions,5298,70047,0.075635,5188,70439,0.073652,-2.621211,-1.413727,0.157442,False


In [ ]:
# files.download(output_path)

### **Additional Links**

[**CSV Table Link**](https://drive.google.com/file/d/1AmWBSfpIsVkuVQtDN4sOjAC64yXtHnDZ/view?usp=sharing)

[**Tableau Link**](https://public.tableau.com/app/profile/yaryna.rachkovska/viz/ABTestConversionLiftStatisticalSignificance/ABTestOverview?publish=yes)

### **Results & Key Insights**

Across the four A/B tests, the test variant shows mixed performance across the funnel, with no consistent overall uplift.

**Test 1** delivered the strongest positive impact, with significant lifts in Add Payment Info, Add Shipping Info, and Begin Checkout.

**Test 2** showed modest positive lifts across all funnel events, but none were statistically significant.

In **Tests 3 and 4**, performance declined at the lower-funnel stages, particularly at Begin Checkout.

Begin Checkout showed a statistically significant negative effect in both tests, indicating a potentially meaningful deterioration in conversion.

**Test 4** also showed a significant decline in New Accounts.

Overall, the results suggest that the tested change can improve some upper-funnel actions, but its impact is not consistently sustained through the funnel.

Further investigation should focus on the factors driving the negative impact on checkout and account creation before scaling the tested variant.